## Connect5 AI Arena – Policy Network Training
We train a CNN to predict the winning player’s chosen column from a board state.
Input: (2, 9, 10) tensor (current player’s pieces & opponent’s pieces).
Output: probability distribution over 10 columns, restricted to legal moves.

The dataset contains only moves from episodes won by the player who made the move (i.e., "expert" data).

In [ ]:
# 1. Imports & configuration
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

# Global config
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 256
EPOCHS = 50
LR = 1e-3
ROWS, COLS = 9, 10

DATA_PATH = Path('../data/processed/policy_network_data.npz')
MODEL_DIR = Path('../ml/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

%matplotlib inline

In [ ]:
# 2. Load and prepare data
data = np.load(DATA_PATH)
states = data['states']          # (N, 2, 9, 10) float32
actions = data['actions']        # (N,) int32
valid_masks = data['valid_masks']  # (N, 10) bool

print('Raw shapes:')
print('  states:', states.shape)
print('  actions:', actions.shape)
print('  valid_masks:', valid_masks.shape)
print('  Unique actions:', np.unique(actions))

# Train/validation split
X_train, X_val, y_train, y_val, m_train, m_val = train_test_split(
    states, actions, valid_masks, test_size=0.15, random_state=42, stratify=actions
)

print(f'Train size: {len(X_train)}, Validation size: {len(X_val)}')

In [ ]:
# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train, dtype=torch.long)
m_train_t = torch.tensor(m_train, dtype=torch.bool)

X_val_t = torch.tensor(X_val)
y_val_t = torch.tensor(y_val, dtype=torch.long)
m_val_t = torch.tensor(m_val, dtype=torch.bool)

train_dataset = TensorDataset(X_train_t, y_train_t, m_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t, m_val_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# 3. Model definition – lightweight CNN with residual blocks
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)

class PolicyNetwork(nn.Module):
    def __init__(self, input_channels=2, rows=9, cols=10, num_actions=10):
        super().__init__()
        self.rows, self.cols = rows, cols
        # Initial conv block
        self.conv_init = nn.Sequential(
            nn.Conv2d(input_channels, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        # Two residual blocks
        self.res1 = ResidualBlock(128)
        self.res2 = ResidualBlock(128)
        
        # Policy head – first global avg pool, then conv to action logits
        self.policy_conv = nn.Conv2d(128, 2, kernel_size=1)  # reduce to 2 channels
        self.policy_fc = nn.Linear(2 * rows * cols, num_actions)

    def forward(self, x):
        # x: (batch, 2, 9, 10)
        out = self.conv_init(x)
        out = self.res1(out)
        out = self.res2(out)
        
        # Policy logits
        p = self.policy_conv(out)
        p = p.reshape(p.size(0), -1)  # flatten
        logits = self.policy_fc(p)
        return logits  # raw logits, we will mask manually in loss

# Instantiate model
model = PolicyNetwork(input_channels=2, rows=ROWS, cols=COLS, num_actions=COLS).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
# 4. Masked cross-entropy loss and accuracy
def masked_cross_entropy(logits, target, mask):
    """
    logits: (batch, num_actions)
    target: (batch,)
    mask  : (batch, num_actions) boolean
    Only consider logits of legal actions for the loss.
    """
    # Set logits of illegal actions to a very negative value so softmax ~ 0
    masked_logits = logits.clone()
    masked_logits[~mask] = float('-inf')
    return F.cross_entropy(masked_logits, target)

def masked_accuracy(logits, target, mask):
    """
    Accuracy considering only legal moves.
    """
    masked_logits = logits.clone()
    masked_logits[~mask] = float('-inf')
    pred = masked_logits.argmax(dim=1)
    return (pred == target).float().mean().item()

In [ ]:
# 5. Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    for Xb, yb, mb in train_loader:
        Xb, yb, mb = Xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = masked_cross_entropy(logits, yb, mb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * Xb.size(0)
        total_acc += masked_accuracy(logits, yb, mb) * Xb.size(0)
    scheduler.step()

    avg_train_loss = total_loss / len(train_dataset)
    avg_train_acc = total_acc / len(train_dataset)
    train_losses.append(avg_train_loss)
    train_accs.append(avg_train_acc)

    # Validation
    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for Xb, yb, mb in val_loader:
            Xb, yb, mb = Xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
            logits = model(Xb)
            loss = masked_cross_entropy(logits, yb, mb)
            val_loss += loss.item() * Xb.size(0)
            val_acc += masked_accuracy(logits, yb, mb) * Xb.size(0)
    val_loss /= len(val_dataset)
    val_acc /= len(val_dataset)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} Acc: {avg_train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_DIR / 'policy_net_best.pth')
        print(f"  -> saved new best model")

print(f"Training complete. Best val accuracy: {best_val_acc:.4f}")

In [ ]:
# 6. Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train')
ax1.plot(val_losses, label='Val')
ax1.set_title('Loss')
ax1.legend()
ax2.plot(train_accs, label='Train')
ax2.plot(val_accs, label='Val')
ax2.set_title('Accuracy')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7. Evaluate on a few validation samples interactively
model.eval()
with torch.no_grad():
    idx = np.random.randint(0, len(X_val))
    sample_state = X_val_t[idx].unsqueeze(0).to(DEVICE)
    sample_mask = m_val_t[idx].unsqueeze(0).to(DEVICE)
    sample_target = y_val_t[idx].item()
    logits = model(sample_state)
    masked_logits = logits.clone()
    masked_logits[~sample_mask] = float('-inf')
    probs = F.softmax(masked_logits, dim=1).squeeze().cpu().numpy()
    print(f"True action: {sample_target}")
    print(f"Predicted probabilities (legal moves only):")
    for col in range(COLS):
        if sample_mask[0, col]:
            print(f"  Column {col}: {probs[col]:.4f}")
    # Show the board as text
    board_np = sample_state.squeeze().cpu().numpy()  # (2,9,10)
    print("\nBoard (channel 0 = current player, channel 1 = opponent):\n")
    for row in reversed(range(ROWS)):  # top to bottom visual
        line = ""
        for col in range(COLS):
            if board_np[0, row, col] == 1:
                line += "X"
            elif board_np[1, row, col] == 1:
                line += "O"
            else:
                line += "."
        print(row, line)
    print("   ", "".join(str(c) for c in range(COLS)))

In [ ]:
# 8. Export to ONNX for backend inference
import torch.onnx

# Load best weights
model.load_state_dict(torch.load(MODEL_DIR / 'policy_net_best.pth', map_location=DEVICE))
model.eval()

dummy_input = torch.randn(1, 2, ROWS, COLS, device=DEVICE)
onnx_path = MODEL_DIR / 'policy_net.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['board'],
    output_names=['logits'],
    dynamic_axes={'board': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=13
)
print(f"Model exported to {onnx_path}")

# Quick sanity check with ONNX Runtime
import onnxruntime as ort
session = ort.InferenceSession(str(onnx_path))
inputs = {session.get_inputs()[0].name: dummy_input.cpu().numpy()}
out = session.run(None, inputs)[0]
print(f"ONNX output shape: {out.shape}")